# CS5760 Homework 2

Student: Srinivas Reddy Pittu  
ID: 700777009  
University: University of Central Missouri  
Department: Computer Science and Cybersecurity  
Course: CS5760 Natural Language Processing  
Semester: Fall 2026

This assignment covers Naive Bayes document classification, harms of classification, bigram probabilities, add-1 smoothing, trigram-to-bigram backoff, and multiclass evaluation.


Open in Google Colab and select **Runtime > Run all**. No installations are required.


## Part I Q5 Python implementation

This cell accepts the matrix through `calculate_metrics(matrix, labels)`. Rows are predictions and columns are gold labels.


In [1]:
"""CS5760 Homework 2, Part I Q5 - Srinivas Reddy Pittu."""


def calculate_metrics(matrix, labels):
    """Rows are system predictions; columns are gold labels."""
    n = len(labels)
    if n == 0 or len(matrix) != n or any(len(r) != n for r in matrix):
        raise ValueError("Provide a square matrix and one label per class.")
    if any(not isinstance(x, int) or x < 0 for r in matrix for x in r):
        raise ValueError("Counts must be nonnegative integers.")

    results = []
    total_tp = total_fp = total_fn = 0
    for i, label in enumerate(labels):
        tp = matrix[i][i]
        fp = sum(matrix[i]) - tp  # Other gold labels in this row.
        fn = sum(row[i] for row in matrix) - tp  # Other predictions.
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        results.append((label, tp, fp, fn, precision, recall))
        total_tp += tp
        total_fp += fp
        total_fn += fn

    # Macro averages give each class equal weight.
    macro_p = sum(r[4] for r in results) / n
    macro_r = sum(r[5] for r in results) / n
    # Micro averages pool the counts before dividing.
    pred_total = total_tp + total_fp
    gold_total = total_tp + total_fn
    micro_p = total_tp / pred_total if pred_total else 0.0
    micro_r = total_tp / gold_total if gold_total else 0.0
    return results, macro_p, macro_r, micro_p, micro_r


def main():
    # Replace these inputs to evaluate a different confusion matrix.
    matrix = [[5, 10, 5], [15, 20, 10], [0, 15, 10]]
    labels = ["Cat", "Dog", "Rabbit"]
    rows, macro_p, macro_r, micro_p, micro_r = calculate_metrics(
        matrix, labels
    )
    print("Rows = system predictions; columns = gold labels")
    print("Class      TP  FP  FN   Precision    Recall")
    for label, tp, fp, fn, precision, recall in rows:
        print(f"{label:<9} {tp:2}  {fp:2}  {fn:2}   "
              f"{precision:.6f}    {recall:.6f}")
    print(f"Macro precision: {macro_p:.6f}")
    print(f"Macro recall:    {macro_r:.6f}")
    print(f"Micro precision: {micro_p:.6f}")
    print(f"Micro recall:    {micro_r:.6f}")


if __name__ == "__main__":
    main()


Rows = system predictions; columns = gold labels
Class      TP  FP  FN   Precision    Recall
Cat        5  15  15   0.250000    0.250000
Dog       20  25  25   0.444444    0.444444
Rabbit    10  15  15   0.400000    0.400000
Macro precision: 0.364815
Macro recall:    0.364815
Micro precision: 0.388889
Micro recall:    0.388889


## Part II Q1 Bigram language model

The program reads the three training sentences, prints all counts and MLE probabilities, and compares both test sentences. Sentence boundaries are included, capitalization is preserved, and unseen transitions receive zero. Exact fractions prevent rounding in the probability products.


In [2]:
"""CS5760 Homework 2, Part II Q1 - Srinivas Reddy Pittu."""
from collections import Counter
from fractions import Fraction


CORPUS = [
    "<s> I love NLP </s>",
    "<s> I love deep learning </s>",
    "<s> deep learning is fun </s>",
]


def tokenize(sentence):
    """Split on spaces; add one start and end marker if needed."""
    words = sentence.split()
    if words and words[0] == "<s>":
        words = words[1:]
    if words and words[-1] == "</s>":
        words = words[:-1]
    if "<s>" in words or "</s>" in words:
        raise ValueError("Boundary markers must be at sentence ends.")
    return ["<s>"] + words + ["</s>"]


def train(corpus):
    """Count tokens and adjacent pairs without crossing sentences."""
    unigrams, bigrams, contexts = Counter(), Counter(), Counter()
    for sentence in corpus:
        tokens = tokenize(sentence)
        unigrams.update(tokens)
        bigrams.update(zip(tokens, tokens[1:]))
        contexts.update(tokens[:-1])  # Only words with a successor.
    return unigrams, bigrams, contexts


def bigram_probability(previous, word, bigrams, contexts):
    """MLE: count(previous, word) / count(previous with successor)."""
    count = contexts.get(previous, 0)
    if count == 0:
        return Fraction(0, 1)
    return Fraction(bigrams.get((previous, word), 0), count)


def sentence_probability(sentence, bigrams, contexts):
    """Include the end marker; unseen transitions give zero in MLE."""
    tokens = tokenize(sentence)
    score = Fraction(1, 1)  # Exact arithmetic avoids underflow.
    for previous, word in zip(tokens, tokens[1:]):
        score *= bigram_probability(previous, word, bigrams, contexts)
    return score


def main():
    unigrams, bigrams, contexts = train(CORPUS)
    print("Unigram counts (including boundary markers):")
    for word, count in sorted(unigrams.items()):
        print(f"  {word}: {count}")
    print("\nBigram counts and MLE probabilities:")
    for (previous, word), count in sorted(bigrams.items()):
        p = bigram_probability(previous, word, bigrams, contexts)
        print(f"  ({previous}, {word}): count={count}, "
              f"P={p} ({float(p):.6f})")

    tests = CORPUS[:2]
    scores = []
    for sentence in tests:
        tokens = tokenize(sentence)
        factors = [bigram_probability(a, b, bigrams, contexts)
                   for a, b in zip(tokens, tokens[1:])]
        score = sentence_probability(sentence, bigrams, contexts)
        scores.append(score)
        print(f"\nSentence: {sentence}")
        print("  Factors: " + " * ".join(map(str, factors)))
        print(f"  Probability: {score} = {float(score):.6f}")

    if scores[0] == scores[1]:
        print("\nThe model gives both sentences the same probability.")
    else:
        best = 0 if scores[0] > scores[1] else 1
        print(f"\nPreferred sentence: {tests[best]}")
        print("S1 has probability 1/3; S2 has probability 1/6.")
        print("P(NLP | love) and P(deep | love) both equal 1/2.")
        print("S2 also has P(</s> | learning) = 1/2.")
        print("P(</s> | NLP) = 1, so S1 is twice as probable.")


if __name__ == "__main__":
    main()


Unigram counts (including boundary markers):
  </s>: 3
  <s>: 3
  I: 2
  NLP: 1
  deep: 2
  fun: 1
  is: 1
  learning: 2
  love: 2

Bigram counts and MLE probabilities:
  (<s>, I): count=2, P=2/3 (0.666667)
  (<s>, deep): count=1, P=1/3 (0.333333)
  (I, love): count=2, P=1 (1.000000)
  (NLP, </s>): count=1, P=1 (1.000000)
  (deep, learning): count=2, P=1 (1.000000)
  (fun, </s>): count=1, P=1 (1.000000)
  (is, fun): count=1, P=1 (1.000000)
  (learning, </s>): count=1, P=1/2 (0.500000)
  (learning, is): count=1, P=1/2 (0.500000)
  (love, NLP): count=1, P=1/2 (0.500000)
  (love, deep): count=1, P=1/2 (0.500000)

Sentence: <s> I love NLP </s>
  Factors: 2/3 * 1 * 1/2 * 1
  Probability: 1/3 = 0.333333

Sentence: <s> I love deep learning </s>
  Factors: 2/3 * 1 * 1/2 * 1 * 1/2
  Probability: 1/6 = 0.166667

Preferred sentence: <s> I love NLP </s>
S1 has probability 1/3; S2 has probability 1/6.
P(NLP | love) and P(deep | love) both equal 1/2.
S2 also has P(</s> | learning) = 1/2.
P(</s> | NL

## Sources

Homework 2.docx and the supplied Text Classification and Naive Bayes lecture PDF. See PDF pages 27-29, 36-37, 54, 63, 69, 75-79, and 82-85.
